# Fondasi Machine Learning

## Bagian 2e: Lab Regularisasi

### Tujuan Pembelajaran

- menerapkan standardisasi data;
- membandingkan koefisien model dengan dan tanpa standardisasi;
- menerapkan Lasso dan Ridge serta membaca pengaruh alpha;
- menilai model pada data uji.

In [ ]:
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings("ignore")
np.set_printoptions(precision=3, suppress=True)

### Memuat data

Data harga rumah Boston diunduh sekali, lalu dibaca dari berkas pickle.

In [ ]:
data_path = Path("data/boston_housing_clean.pickle")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/boston_housing_clean.pickle"

if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(data_url, data_path)

bundle = pd.read_pickle(data_path)
housing = bundle["dataframe"]
housing.head()

## Standardisasi data

Standardisasi mengubah setiap variabel agar rata-ratanya nol dan simpangan bakunya satu. Tanpa langkah ini, fitur berskala besar mendominasi penalti regularisasi.

In [ ]:
target_column = "MEDV"

features = housing.drop(columns=target_column)
target = housing[target_column]

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

### Latihan: verifikasi hasil standardisasi

Hitung standardisasi secara manual memakai rumus (x - rata-rata) / simpangan baku, lalu bandingkan dengan keluaran `StandardScaler`.

In [ ]:
values = np.array(features)
manual_scaled = (values - values.mean(axis=0)) / values.std(axis=0)
np.allclose(manual_scaled, features_scaled)

Hasilnya identik, sehingga standardisasi memang hanya pengurangan rata-rata dan pembagian simpangan baku per kolom.

## Koefisien dengan dan tanpa standardisasi

Koefisien regresi linear bergantung pada skala fitur. Fitur berskala kecil mendapat koefisien besar meskipun pengaruhnya belum tentu besar, sehingga koefisien antarfitur tidak bisa dibandingkan langsung.

In [ ]:
plain_model = LinearRegression().fit(features, target)
print("Koefisien tanpa standardisasi:")
print(plain_model.coef_)

In [ ]:
scaled_model = LinearRegression().fit(features_scaled, target)
print("Koefisien setelah standardisasi:")
print(scaled_model.coef_)

### Latihan: fitur paling berpengaruh

Setelah standardisasi, besar koefisien bisa dibandingkan antarfitur. Urutkan fitur berdasarkan nilai absolut koefisiennya.

In [ ]:
importance = pd.DataFrame({
    "feature": features.columns,
    "coefficient": scaled_model.coef_,
})
importance["abs_coefficient"] = importance["coefficient"].abs()
importance.sort_values("abs_coefficient", ascending=False).head(10)

Dengan koefisien yang sudah setara skala, `LSTAT`, `RM`, `DIS`, dan `RAD` termasuk fitur paling berpengaruh. scikit-learn juga menyediakan cara bawaan untuk memeringkat fitur, tetapi tabel di atas cukup untuk membaca arah dan besarnya pengaruh.

## Lasso dengan dan tanpa standardisasi

Lasso menambahkan penalti L1 yang mendorong sebagian koefisien menjadi nol. Karena penalti bekerja pada besar koefisien, skalanya sangat menentukan fitur mana yang bertahan.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
features_poly = poly.fit_transform(features)

poly_scaler = StandardScaler()
features_poly_scaled = poly_scaler.fit_transform(features_poly)

print("Bentuk fitur polinomial:", features_poly_scaled.shape)

In [ ]:
lasso = Lasso().fit(features_poly_scaled, target)
print("Jumlah koefisien:", len(lasso.coef_))
print("Jumlah koefisien tidak nol:", int((lasso.coef_ != 0).sum()))

### Latihan: pengaruh alpha pada Lasso

Bandingkan jumlah koefisien nol dan total besar koefisien untuk alpha 0,1 dan alpha 1.

In [ ]:
lasso_small = Lasso(alpha=0.1).fit(features_poly_scaled, target)
print("alpha 0.1 - total |koefisien|:", round(np.abs(lasso_small.coef_).sum(), 3))
print("alpha 0.1 - koefisien tidak nol:", int((lasso_small.coef_ != 0).sum()))

In [ ]:
lasso_large = Lasso(alpha=1).fit(features_poly_scaled, target)
print("alpha 1.0 - total |koefisien|:", round(np.abs(lasso_large.coef_).sum(), 3))
print("alpha 1.0 - koefisien tidak nol:", int((lasso_large.coef_ != 0).sum()))

Semakin besar alpha, semakin besar penalti sehingga total besar koefisien turun dan lebih banyak koefisien menjadi nol. Model jadi lebih sederhana, tetapi berisiko kehilangan informasi bila alpha terlalu besar.

### Latihan: R kuadrat pada data latih

Hitung R kuadrat model Lasso tanpa pembagian data latih dan uji.

In [ ]:
r2_score(target, lasso.predict(features_poly_scaled))

Regularisasi tidak akan memperbaiki skor bila dievaluasi pada data yang sama dengan data pelatihan, karena penalti justru membatasi model. Manfaatnya baru terlihat pada data baru.

## Dengan pembagian data latih dan uji

Standardisasi harus di-fit pada data latih saja, lalu diterapkan ke data uji. Bila scaler di-fit pada seluruh data, informasi data uji bocor ke proses pelatihan dan skor uji menjadi terlalu optimistis.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features_poly, target, test_size=0.3, random_state=72018
)

split_scaler = StandardScaler()
X_train_scaled = split_scaler.fit_transform(X_train)
X_test_scaled = split_scaler.transform(X_test)

In [ ]:
for alpha in [1.0, 0.1, 0.001]:
    model = Lasso(alpha=alpha, max_iter=100000).fit(X_train_scaled, y_train)
    score = r2_score(y_test, model.predict(X_test_scaled))
    print(f"alpha {alpha:<6} R2 uji {score:.4f} | koefisien tidak nol {int((model.coef_ != 0).sum())}")

Alpha kecil memberi R kuadrat uji terbaik karena penalti hampir tidak berpengaruh, tetapi jumlah koefisien yang bertahan juga paling banyak. Alpha besar menyederhanakan model dengan sedikit penurunan skor.

### Latihan: bandingkan dengan regresi linear

Latih regresi linear tanpa penalti pada data yang sama, lalu bandingkan total besar koefisiennya dengan Lasso alpha 0,001.

In [ ]:
lasso_tiny = Lasso(alpha=0.001, max_iter=100000).fit(X_train_scaled, y_train)
linear_baseline = LinearRegression().fit(X_train_scaled, y_train)

print("R2 uji Lasso alpha 0.001:", round(r2_score(y_test, lasso_tiny.predict(X_test_scaled)), 4))
print("R2 uji regresi linear    :", round(r2_score(y_test, linear_baseline.predict(X_test_scaled)), 4))
print("Total |koefisien| Lasso  :", round(np.abs(lasso_tiny.coef_).sum(), 3))
print("Total |koefisien| linear :", round(np.abs(linear_baseline.coef_).sum(), 3))
print("Koefisien nol Lasso      :", int((lasso_tiny.coef_ == 0).sum()))

Pada alpha yang sangat kecil, skor Lasso bahkan sedikit lebih tinggi daripada regresi linear. Penalti tetap menolkan sebagian koefisien, sehingga modelnya lebih ringkas tanpa kehilangan akurasi.

## Regularisasi L1 dan L2

Lasso memakai penalti L1 yang menghasilkan koefisien tepat nol. Ridge memakai penalti L2 yang hanya menyusutkan koefisien tanpa menghapusnya. Sintaks keduanya sama di scikit-learn.

In [ ]:
ridge_tiny = Ridge(alpha=0.001).fit(X_train_scaled, y_train)
ridge_predictions = ridge_tiny.predict(X_test_scaled)

print("R2 uji Ridge alpha 0.001:", round(r2_score(y_test, ridge_predictions), 4))
print("Total |koefisien| Ridge :", round(np.abs(ridge_tiny.coef_).sum(), 3))
print("Total |koefisien| Lasso :", round(np.abs(lasso_tiny.coef_).sum(), 3))
print("Koefisien nol Ridge     :", int((ridge_tiny.coef_ == 0).sum()))
print("Koefisien nol Lasso     :", int((lasso_tiny.coef_ == 0).sum()))

Ridge tidak pernah menghasilkan koefisien nol, sedangkan Lasso menolkan sebagian koefisien. Pada dataset ini total besar koefisien Lasso lebih kecil, tanda penalti L1 lebih agresif menyusutkan.

In [ ]:
print("R2 Ridge pada seluruh data:", round(r2_score(target, ridge_tiny.predict(features_poly_scaled)), 4))
print("R2 Lasso pada seluruh data:", round(r2_score(target, lasso_tiny.predict(features_poly_scaled)), 4))

Tanpa memperhitungkan overfitting, Lasso pada alpha 0,001 sedikit lebih baik daripada Ridge pada alpha yang sama, meski keduanya hanya berbeda tipis.

## Apakah urutan penskalaan berpengaruh?

Untuk regresi linear, hasil prediksi sama saja apakah data diskalakan sebelum atau sesudah pembagian data, selama scaler hanya di-fit pada data latih. Yang berubah hanyalah nilai koefisien, bukan prediksinya.

In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    features, target, test_size=0.3, random_state=72018
)

raw_model = LinearRegression().fit(X_train_raw, y_train_raw)
raw_predictions = raw_model.predict(X_test_raw)
print("R2 tanpa standardisasi:", round(r2_score(y_test_raw, raw_predictions), 4))

In [ ]:
scaled_raw_model = LinearRegression()
raw_scaler = StandardScaler()
X_train_raw_scaled = raw_scaler.fit_transform(X_train_raw)
scaled_raw_model.fit(X_train_raw_scaled, y_train_raw)
scaled_predictions = scaled_raw_model.predict(raw_scaler.transform(X_test_raw))

print("R2 dengan standardisasi:", round(r2_score(y_test_raw, scaled_predictions), 4))
print("Prediksi identik:", np.allclose(raw_predictions, scaled_predictions))

## Ringkasan

- Standardisasi membuat koefisien antarfitur bisa dibandingkan.
- Lasso (L1) menolkan sebagian koefisien, Ridge (L2) hanya menyusutkannya.
- Alpha mengatur kekuatan penalti: makin besar, makin sederhana modelnya.
- Scaler harus di-fit pada data latih saja untuk menghindari kebocoran data.